In [ ]:
import gc
import os
import time
import warnings
from itertools import combinations
from warnings import simplefilter
import polars as pl
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit
import catboost as cbt

warnings.filterwarnings("ignore")
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

is_offline = True
is_train = True
is_infer = True
max_lookback = np.nan
split_day = 435

In [26]:
df = pd.read_csv(r'C:\Users\cwang\Desktop\Kaggle_optiver\train.csv')
df = df.dropna(subset=["target"])
df.reset_index(drop=True, inplace=True)
df.shape

(5237892, 17)

In [27]:
from sklearn.model_selection import KFold
from sklearn.model_selection._split import _BaseKFold, indexable, _num_samples
from sklearn.utils.validation import _deprecate_positional_args

class PurgedGroupTimeSeriesSplit(_BaseKFold):
    """Time Series cross-validator variant with non-overlapping groups.
    Allows for a gap in groups to avoid potentially leaking info from
    train into test if the model has windowed or lag features.
    Provides train/test indices to split time series data samples
    that are observed at fixed time intervals according to a
    third-party provided group.
    In each split, test indices must be higher than before, and thus shuffling
    in cross validator is inappropriate.
    This cross-validation object is a variation of :class:`KFold`.
    In the kth split, it returns first k folds as train set and the
    (k+1)th fold as test set.
    The same group will not appear in two different folds (the number of
    distinct groups has to be at least equal to the number of folds).
    Note that unlike standard cross-validation methods, successive
    training sets are supersets of those that come before them.
    Read more in the :ref:`User Guide <cross_validation>`.
    Parameters
    ----------
    n_splits : int, default=5
        Number of splits. Must be at least 2.
    max_train_group_size : int, default=Inf
        Maximum group size for a single training set.
    group_gap : int, default=None
        Gap between train and test
    max_test_group_size : int, default=Inf
        We discard this number of groups from the end of each train split
    """

    @_deprecate_positional_args
    def __init__(self,
                 n_splits=5,
                 *,
                 max_train_group_size=np.inf,
                 max_test_group_size=np.inf,
                 group_gap=None,
                 verbose=False
                 ):
        super().__init__(n_splits, shuffle=False, random_state=None)
        self.max_train_group_size = max_train_group_size
        self.group_gap = group_gap
        self.max_test_group_size = max_test_group_size
        self.verbose = verbose

    def split(self, X, y=None, groups=None):
        """Generate indices to split data into training and test set.
        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training data, where n_samples is the number of samples
            and n_features is the number of features.
        y : array-like of shape (n_samples,)
            Always ignored, exists for compatibility.
        groups : array-like of shape (n_samples,)
            Group labels for the samples used while splitting the dataset into
            train/test set.
        Yields
        ------
        train : ndarray
            The training set indices for that split.
        test : ndarray
            The testing set indices for that split.
        """
        if groups is None:
            raise ValueError(
                "The 'groups' parameter should not be None")
        X, y, groups = indexable(X, y, groups)
        n_samples = _num_samples(X)
        n_splits = self.n_splits
        group_gap = self.group_gap
        max_test_group_size = self.max_test_group_size
        max_train_group_size = self.max_train_group_size
        n_folds = n_splits + 1
        group_dict = {}
        u, ind = np.unique(groups, return_index=True)
        unique_groups = u[np.argsort(ind)]
        n_samples = _num_samples(X)
        n_groups = _num_samples(unique_groups)
        for idx in np.arange(n_samples):
            if (groups[idx] in group_dict):
                group_dict[groups[idx]].append(idx)
            else:
                group_dict[groups[idx]] = [idx]
        if n_folds > n_groups:
            raise ValueError(
                ("Cannot have number of folds={0} greater than"
                 " the number of groups={1}").format(n_folds,
                                                     n_groups))

        group_test_size = min(n_groups // n_folds, max_test_group_size)
        group_test_starts = range(n_groups - n_splits * group_test_size,
                                  n_groups, group_test_size)
        for group_test_start in group_test_starts:
            train_array = []
            test_array = []

            group_st = max(0, group_test_start - group_gap - max_train_group_size)
            for train_group_idx in unique_groups[group_st:(group_test_start - group_gap)]:
                train_array_tmp = group_dict[train_group_idx]
                
                train_array = np.sort(np.unique(
                                      np.concatenate((train_array,
                                                      train_array_tmp)),
                                      axis=None), axis=None)

            train_end = train_array.size
 
            for test_group_idx in unique_groups[group_test_start:
                                                group_test_start +
                                                group_test_size]:
                test_array_tmp = group_dict[test_group_idx]
                test_array = np.sort(np.unique(
                                              np.concatenate((test_array,
                                                              test_array_tmp)),
                                     axis=None), axis=None)

            test_array  = test_array[group_gap:]
            
            
            if self.verbose > 0:
                    pass
                    
            yield [int(i) for i in train_array], [int(i) for i in test_array]

In [28]:
def reduce_mem_usage(df, verbose=0):
    """
    Iterate through all numeric columns of a dataframe and modify the data type
    to reduce memory usage.
    """

    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float32)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float32)

    if verbose:
        logger.info(f"Memory usage of dataframe is {start_mem:.2f} MB")
        end_mem = df.memory_usage().sum() / 1024**2
        logger.info(f"Memory usage after optimization is: {end_mem:.2f} MB")
        decrease = 100 * (start_mem - end_mem) / start_mem
        logger.info(f"Decreased by {decrease:.2f}%")

    return df

In [29]:
from numba import njit, prange

@njit(parallel=True)
def compute_triplet_imbalance(df_values, comb_indices):
    num_rows = df_values.shape[0]
    num_combinations = len(comb_indices)
    imbalance_features = np.empty((num_rows, num_combinations))

    for i in prange(num_combinations):
        a, b, c = comb_indices[i]
        for j in range(num_rows):
            max_val = max(df_values[j, a], df_values[j, b], df_values[j, c])
            min_val = min(df_values[j, a], df_values[j, b], df_values[j, c])
            mid_val = df_values[j, a] + df_values[j, b] + df_values[j, c] - min_val - max_val
            if mid_val == min_val:  # Prevent division by zero
                imbalance_features[j, i] = np.nan
            else:
                imbalance_features[j, i] = (max_val - mid_val) / (mid_val - min_val)

    return imbalance_features


def calculate_triplet_imbalance_numba(price, df):
    # Convert DataFrame to numpy array for Numba compatibility
    df_values = df[price].values
    comb_indices = [(price.index(a), price.index(b), price.index(c)) for a, b, c in combinations(price, 3)]

    # Calculate the triplet imbalance
    features_array = compute_triplet_imbalance(df_values, comb_indices)

    # Create a DataFrame from the results
    columns = [f"{a}_{b}_{c}_imb2" for a, b, c in combinations(price, 3)]
    features = pd.DataFrame(features_array, columns=columns)

    return features

In [30]:
# # generate imbalance features
# def imbalance_features(df):
#     prices = ["reference_price", "far_price", "near_price", "ask_price", "bid_price", "wap"]
#     sizes = ["matched_size", "bid_size", "ask_size", "imbalance_size"]

#     # V1
#     df["volume"] = df.eval("ask_size + bid_size")
#     df["mid_price"] = df.eval("(ask_price + bid_price) / 2")
#     df["liquidity_imbalance"] = df.eval("(bid_size-ask_size)/(bid_size+ask_size)")
#     df["matched_imbalance"] = df.eval("(imbalance_size-matched_size)/(matched_size+imbalance_size)")
#     df["size_imbalance"] = df.eval("bid_size / ask_size")
    
#     for c in combinations(prices, 2):
#         df[f"{c[0]}_{c[1]}_imb"] = df.eval(f"({c[0]} - {c[1]})/({c[0]} + {c[1]})")

#     for c in [['ask_price', 'bid_price', 'wap', 'reference_price'], sizes]:
#         triplet_feature = calculate_triplet_imbalance_numba(c, df)
#         df[triplet_feature.columns] = triplet_feature.values
        
#     # V2
#     df["stock_weights"] = df["stock_id"].map(weights)
#     df["weighted_wap"] = df["stock_weights"] * df["wap"]
#     df['wap_momentum'] = df.groupby('stock_id')['weighted_wap'].pct_change(periods=6)
#     df["imbalance_momentum"] = df.groupby(['stock_id'])['imbalance_size'].diff(periods=1) / df['matched_size']
#     df["price_spread"] = df["ask_price"] - df["bid_price"]
#     df["spread_intensity"] = df.groupby(['stock_id'])['price_spread'].diff()
#     df['price_pressure'] = df['imbalance_size'] * (df['ask_price'] - df['bid_price'])
#     df['market_urgency'] = df['price_spread'] * df['liquidity_imbalance']
#     df['depth_pressure'] = (df['ask_size'] - df['bid_size']) * (df['far_price'] - df['near_price'])
#     df['spread_depth_ratio'] = (df['ask_price'] - df['bid_price']) / (df['bid_size'] + df['ask_size'])
#     df['mid_price_movement'] = df['mid_price'].diff(periods=5).apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
#     df['micro_price'] = ((df['bid_price'] * df['ask_size']) + (df['ask_price'] * df['bid_size'])) / (df['bid_size'] + df['ask_size'])
#     df['relative_spread'] = (df['ask_price'] - df['bid_price']) / df['wap']
    
#     for func in ["mean", "std", "skew", "kurt"]:
#         df[f"all_prices_{func}"] = df[prices].agg(func, axis=1)
#         df[f"all_sizes_{func}"] = df[sizes].agg(func, axis=1)
        
#     # V3
#     for col in ['matched_size', 'imbalance_size', 'reference_price', 'imbalance_buy_sell_flag']:
#         for window in [1, 2, 3, 5, 10]:
#             df[f"{col}_shift_{window}"] = df.groupby('stock_id')[col].shift(window)
#             df[f"{col}_ret_{window}"] = df.groupby('stock_id')[col].pct_change(window)
            
#     for col in ['ask_price', 'bid_price', 'ask_size', 'bid_size',
#                 'wap', 'near_price', 'far_price']:
#         for window in [1, 2, 3, 5, 10]:
#             df[f"{col}_diff_{window}"] = df.groupby("stock_id")[col].diff(window)

#     return df.replace([np.inf, -np.inf], 0)

# # generate time & stock features
# def other_features(df):
#     df["dow"] = df["date_id"] % 5
#     df["dom"] = df["date_id"] % 20
#     df["seconds"] = df["seconds_in_bucket"] % 60
#     df["minute"] = df["seconds_in_bucket"] // 60

#     for key, value in global_stock_id_feats.items():
#         df[f"global_{key}"] = df["stock_id"].map(value.to_dict())

#     return df

# # generate all features
# def generate_all_features(df):
#     cols = [c for c in df.columns if c not in ["row_id", "time_id", "target"]]
#     df = df[cols]
#     df = imbalance_features(df)
#     df = other_features(df)
#     gc.collect()
    
#     feature_name = [i for i in df.columns if i not in ["row_id", "target", "time_id", "date_id"]]
    
#     return df[feature_name]

In [31]:
def imbalance_features(df):
    # Define lists of price and size-related column names
    prices = ["reference_price", "far_price", "near_price", "ask_price", "bid_price", "wap"]
    sizes = ["matched_size", "bid_size", "ask_size", "imbalance_size"]
    
    df["volume"] = df.eval("ask_size + bid_size")
    df["mid_price"] = df.eval("(ask_price + bid_price) / 2")
    df["liquidity_imbalance"] = df.eval("(bid_size-ask_size)/(bid_size+ask_size)")
    df["matched_imbalance"] = df.eval("(imbalance_size-matched_size)/(matched_size+imbalance_size)")
    df["size_imbalance"] = df.eval("bid_size / ask_size")

    for c in combinations(prices, 2):
        df[f"{c[0]}_{c[1]}_imb"] = df.eval(f"({c[0]} - {c[1]})/({c[0]} + {c[1]})")

    for c in [['ask_price', 'bid_price', 'wap', 'reference_price'], sizes]:
        triplet_feature = calculate_triplet_imbalance_numba(c, df)
        df[triplet_feature.columns] = triplet_feature.values

    df["stock_weights"] = df["stock_id"].map(weights)
    df["weighted_wap"] = df["stock_weights"] * df["wap"]
    ss = df.groupby('time_id')['weighted_wap'].sum()/df.groupby('time_id')['stock_weights'].sum()
    ss = ss.reset_index()
    ss.columns = ['time_id','wapindex']
    df = pd.merge(df,ss,how='left',on='time_id')
    df['wapdiff'] = df['wap'] - df['wapindex']
    
    df['wap_momentum'] = df.groupby('stock_id')['weighted_wap'].pct_change(periods=6)
   
    df["imbalance_momentum"] = df.groupby(['stock_id'])['imbalance_size'].diff(periods=1) / df['matched_size']
    df["price_spread"] = df["ask_price"] - df["bid_price"]
    df["spread_intensity"] = df.groupby(['stock_id'])['price_spread'].diff()
    df['price_pressure'] = df['imbalance_size'] * (df['ask_price'] - df['bid_price'])
    df['market_urgency'] = df['price_spread'] * df['liquidity_imbalance']
    df['depth_pressure'] = (df['ask_size'] - df['bid_size']) * (df['far_price'] - df['near_price'])
    
    df['spread_depth_ratio'] = (df['ask_price'] - df['bid_price']) / (df['bid_size'] + df['ask_size'])
    df['mid_price_movement'] = df['mid_price'].diff(periods=5).apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
    
    df['micro_price'] = ((df['bid_price'] * df['ask_size']) + (df['ask_price'] * df['bid_size'])) / (df['bid_size'] + df['ask_size'])
    df['relative_spread'] = (df['ask_price'] - df['bid_price']) / df['wap']
    
    # Calculate various statistical aggregation features
    for func in ["mean", "std", "skew", "kurt"]:
        df[f"all_prices_{func}"] = df[prices].agg(func, axis=1)
        df[f"all_sizes_{func}"] = df[sizes].agg(func, axis=1)
        

    for col in ['matched_size', 'imbalance_size', 'reference_price', 'imbalance_buy_sell_flag','wapdiff','wap']:
        for window in [1,3,5,10]:
            df[f"{col}_shift_{window}"] = df.groupby('stock_id')[col].shift(window)
            df[f"{col}_ret_{window}"] = df.groupby('stock_id')[col].pct_change(window)
    
    # Calculate diff features for specific columns
    for col in ['ask_price', 'bid_price', 'ask_size', 'bid_size','market_urgency', 'weighted_wap','price_spread']:
        for window in [1,3,5,10]:
            df[f"{col}_diff_{window}"] = df.groupby("stock_id")[col].diff(window)
    
    #V4 feature
    for window in [3,5,10]:
        df[f'price_change_diff_{window}'] = df[f'bid_price_diff_{window}'] - df[f'ask_price_diff_{window}']
        df[f'size_change_diff_{window}'] = df[f'bid_size_diff_{window}'] - df[f'ask_size_diff_{window}']

    #V5 - rolling diff
    # Convert from pandas to Polars
    pl_df = pl.from_pandas(df)

    #Define the windows and columns for which you want to calculate the rolling statistics
    windows = [3, 5, 10]
    columns = ['ask_price', 'bid_price', 'ask_size', 'bid_size']

    # prepare the operations for each column and window
    group = ["stock_id"]
    expressions = []

    # Loop over each window and column to create the rolling mean and std expressions
    for window in windows:
        for col in columns:
            rolling_mean_expr = (
                pl.col(f"{col}_diff_{window}")
                .rolling_mean(window)
                .over(group)
                .alias(f'rolling_diff_{col}_{window}')
            )

            rolling_std_expr = (
                pl.col(f"{col}_diff_{window}")
                .rolling_std(window)
                .over(group)
                .alias(f'rolling_std_diff_{col}_{window}')
            )

            expressions.append(rolling_mean_expr)
            expressions.append(rolling_std_expr)

    # Run the operations using Polars' lazy API
    lazy_df = pl_df.lazy().with_columns(expressions)

    # Execute the lazy expressions and overwrite the pl_df variable
    pl_df = lazy_df.collect()

    # Convert back to pandas if necessary
    df = pl_df.to_pandas()
    gc.collect()
    
    df['mid_price*volume'] = df['mid_price_movement'] * df['volume']
    df['harmonic_imbalance'] = df.eval('2 / ((1 / bid_size) + (1 / ask_size))')
    
    for col in df.columns:
        df[col] = df[col].replace([np.inf, -np.inf], 0)

    return df

def other_features(df):
    df["dow"] = df["date_id"] % 5  # Day of the week
    df["seconds"] = df["seconds_in_bucket"] % 60  
    df["minute"] = df["seconds_in_bucket"] // 60  
    df['time_to_market_close'] = 540 - df['seconds_in_bucket']
    
    for key, value in global_stock_id_feats.items():
        df[f"global_{key}"] = df["stock_id"].map(value.to_dict())

    return df

def generate_all_features(df):
    # Select relevant columns for feature generation
#     cols = [c for c in df.columns if c not in ["row_id", "time_id", "target"]]
#     df = df[cols]
    
    # Generate imbalance features
    df = imbalance_features(df)
    gc.collect() 
    df = other_features(df)
    gc.collect()  
    feature_name = [i for i in df.columns if i not in ["row_id", "target", "time_id", "date_id"]]
    
    return df[feature_name]

In [32]:
weights = [
    0.004, 0.001, 0.002, 0.006, 0.004, 0.004, 0.002, 0.006, 0.006, 0.002, 0.002, 0.008,
    0.006, 0.002, 0.008, 0.006, 0.002, 0.006, 0.004, 0.002, 0.004, 0.001, 0.006, 0.004,
    0.002, 0.002, 0.004, 0.002, 0.004, 0.004, 0.001, 0.001, 0.002, 0.002, 0.006, 0.004,
    0.004, 0.004, 0.006, 0.002, 0.002, 0.04 , 0.002, 0.002, 0.004, 0.04 , 0.002, 0.001,
    0.006, 0.004, 0.004, 0.006, 0.001, 0.004, 0.004, 0.002, 0.006, 0.004, 0.006, 0.004,
    0.006, 0.004, 0.002, 0.001, 0.002, 0.004, 0.002, 0.008, 0.004, 0.004, 0.002, 0.004,
    0.006, 0.002, 0.004, 0.004, 0.002, 0.004, 0.004, 0.004, 0.001, 0.002, 0.002, 0.008,
    0.02 , 0.004, 0.006, 0.002, 0.02 , 0.002, 0.002, 0.006, 0.004, 0.002, 0.001, 0.02,
    0.006, 0.001, 0.002, 0.004, 0.001, 0.002, 0.006, 0.006, 0.004, 0.006, 0.001, 0.002,
    0.004, 0.006, 0.006, 0.001, 0.04 , 0.006, 0.002, 0.004, 0.002, 0.002, 0.006, 0.002,
    0.002, 0.004, 0.006, 0.006, 0.002, 0.002, 0.008, 0.006, 0.004, 0.002, 0.006, 0.002,
    0.004, 0.006, 0.002, 0.004, 0.001, 0.004, 0.002, 0.004, 0.008, 0.006, 0.008, 0.002,
    0.004, 0.002, 0.001, 0.004, 0.004, 0.004, 0.006, 0.008, 0.004, 0.001, 0.001, 0.002,
    0.006, 0.004, 0.001, 0.002, 0.006, 0.004, 0.006, 0.008, 0.002, 0.002, 0.004, 0.002,
    0.04 , 0.002, 0.002, 0.004, 0.002, 0.002, 0.006, 0.02 , 0.004, 0.002, 0.006, 0.02,
    0.001, 0.002, 0.006, 0.004, 0.006, 0.004, 0.004, 0.004, 0.004, 0.002, 0.004, 0.04,
    0.002, 0.008, 0.002, 0.004, 0.001, 0.004, 0.006, 0.004,
]

weights = {int(k):v for k,v in enumerate(weights)}

In [33]:
if is_offline:
    df_train = df[df["date_id"] <= split_day]
    df_valid = df[df["date_id"] > split_day]
    print("Offline mode")
    print(f"train : {df_train.shape}, valid : {df_valid.shape}")
else:
    df_train = df
    print("Online mode")

Offline mode
train : (4742893, 17), valid : (494999, 17)


In [34]:
# if is_train:
#     global_stock_id_feats = {
#         "median_size": df_train.groupby("stock_id")["bid_size"].median() + df_train.groupby("stock_id")["ask_size"].median(),
#         "std_size": df_train.groupby("stock_id")["bid_size"].std() + df_train.groupby("stock_id")["ask_size"].std(),
#         "ptp_size": df_train.groupby("stock_id")["bid_size"].max() - df_train.groupby("stock_id")["bid_size"].min(),
#         "median_price": df_train.groupby("stock_id")["bid_price"].median() + df_train.groupby("stock_id")["ask_price"].median(),
#         "std_price": df_train.groupby("stock_id")["bid_price"].std() + df_train.groupby("stock_id")["ask_price"].std(),
#         "ptp_price": df_train.groupby("stock_id")["bid_price"].max() - df_train.groupby("stock_id")["ask_price"].min(),
#     }
#     if is_offline:
#         df_train_feats = generate_all_features(df_train)
#         print("Build Train Feats Finished.")
#         df_valid_feats = generate_all_features(df_valid)
#         print("Build Valid Feats Finished.")
#         df_valid_feats = reduce_mem_usage(df_valid_feats)
#     else:
#         df_train_feats = generate_all_features(df_train)
#         print("Build Online Train Feats Finished.")

#     df_train_feats = reduce_mem_usage(df_train_feats)

In [35]:
if is_train:
    global_stock_id_feats = {
        "median_size": df_train.groupby("stock_id")["bid_size"].median() + df_train.groupby("stock_id")["ask_size"].median(),
        "std_size": df_train.groupby("stock_id")["bid_size"].std() + df_train.groupby("stock_id")["ask_size"].std(),
        "ptp_size": df_train.groupby("stock_id")["bid_size"].max() - df_train.groupby("stock_id")["bid_size"].min(),
        "median_price": df_train.groupby("stock_id")["bid_price"].median() + df_train.groupby("stock_id")["ask_price"].median(),
        "std_price": df_train.groupby("stock_id")["bid_price"].std() + df_train.groupby("stock_id")["ask_price"].std(),
        "ptp_price": df_train.groupby("stock_id")["bid_price"].max() - df_train.groupby("stock_id")["ask_price"].min(),
    }
    if is_offline:
        df_train_feats = generate_all_features(df_train)
        print("Build Train Feats Finished.")
        df_valid_feats = generate_all_features(df_valid)
        print("Build Valid Feats Finished.")
        df_valid_feats = reduce_mem_usage(df_valid_feats)
    else:
        df_train_feats = generate_all_features(df_train)
        print("Build Online Train Feats Finished.")

    df_train_feats = reduce_mem_usage(df_train_feats)


Build Train Feats Finished.
Build Valid Feats Finished.


In [36]:
corr = df_train_feats.corr()
# Drop highly correlated features (37->30)
columns = np.full((corr.shape[0],), True, dtype=bool)
for i in range(corr.shape[0]):
    for j in range(i+1, corr.shape[0]):
        if corr.iloc[i,j] >= 0.99:
            if columns[j]:
                columns[j] = False

feature_columns = df_train_feats.columns[columns].values
drop_columns = df_train_feats.columns[columns == False].values
print(feature_columns)
print('-'*73)
print(drop_columns)

gc.collect()

['stock_id' 'seconds_in_bucket' 'imbalance_size' 'imbalance_buy_sell_flag'
 'reference_price' 'matched_size' 'far_price' 'near_price' 'bid_price'
 'bid_size' 'ask_price' 'ask_size' 'wap' 'volume' 'liquidity_imbalance'
 'matched_imbalance' 'size_imbalance' 'reference_price_far_price_imb'
 'reference_price_near_price_imb' 'reference_price_ask_price_imb'
 'reference_price_bid_price_imb' 'reference_price_wap_imb'
 'far_price_near_price_imb' 'far_price_ask_price_imb'
 'near_price_ask_price_imb' 'ask_price_bid_price_imb' 'ask_price_wap_imb'
 'bid_price_wap_imb' 'ask_price_bid_price_wap_imb2'
 'ask_price_bid_price_reference_price_imb2'
 'ask_price_wap_reference_price_imb2' 'bid_price_wap_reference_price_imb2'
 'matched_size_bid_size_ask_size_imb2'
 'matched_size_bid_size_imbalance_size_imb2'
 'matched_size_ask_size_imbalance_size_imb2'
 'bid_size_ask_size_imbalance_size_imb2' 'stock_weights' 'wapindex'
 'wapdiff' 'wap_momentum' 'imbalance_momentum' 'spread_intensity'
 'price_pressure' 'market

0

In [37]:
# if is_train:
#     feature_name = list(df_train_feats.columns)
#     lgb_params = {
#         "objective" : "mae",
#         "n_estimators" : 3000,
#         "num_leaves" : 128,
#         "subsample" : 0.6,
#         "colsample_bytree" : 0.6,
#         "learning_rate" : 0.05,
#         "n_jobs" : 4,
#         "device" : "gpu",
#         "verbosity": -1,
#         "importance_type" : "gain",
#     }

#     print(f"Feature length = {len(feature_name)}")

#     offline_split = df_train['date_id']>(split_day - 45)
#     df_offline_train = df_train_feats[~offline_split]
#     df_offline_valid = df_train_feats[offline_split]
#     df_offline_train_target = df_train['target'][~offline_split]
#     df_offline_valid_target = df_train['target'][offline_split]

#     print("Valid Model Trainning.")
#     lgb_model = lgb.LGBMRegressor(**lgb_params)
#     lgb_model.fit(
#         df_offline_train[feature_name],
#         df_offline_train_target,
#         eval_set=[(df_offline_valid[feature_name], df_offline_valid_target)],
#         callbacks=[
#             lgb.callback.early_stopping(stopping_rounds=100),
#             lgb.callback.log_evaluation(period=100),
#         ],
#     )

#     del df_offline_train, df_offline_valid, df_offline_train_target, df_offline_valid_target
#     gc.collect()

#     # infer
#     df_train_target = df_train["target"]
#     print("Infer Model Trainning.")
#     infer_params = lgb_params.copy()
#     infer_params["n_estimators"] = int(1.2 * lgb_model.best_iteration_)
#     infer_lgb_model = lgb.LGBMRegressor(**infer_params)
#     infer_lgb_model.fit(df_train_feats[feature_name], df_train_target)

#     if is_offline:   
#         # offline predictions
#         df_valid_target = df_valid["target"]
#         offline_predictions = infer_lgb_model.predict(df_valid_feats[feature_name])
#         offline_score = mean_absolute_error(offline_predictions, df_valid_target)
#         print(f"Offline Score {np.round(offline_score, 4)}")

In [38]:
df_valid

,stock_id,date_id,seconds_in_bucket,imbalance_size,imbalance_buy_sell_flag,reference_price,matched_size,far_price,near_price,bid_price,...,ask_price_bid_price_wap_imb2,ask_price_bid_price_reference_price_imb2,ask_price_wap_reference_price_imb2,bid_price_wap_reference_price_imb2,matched_size_bid_size_ask_size_imb2,matched_size_bid_size_imbalance_size_imb2,matched_size_ask_size_imbalance_size_imb2,bid_size_ask_size_imbalance_size_imb2,stock_weights,weighted_wap
4742893,0,436,0,0.00,0,1.000268,12874820.16,NaN,NaN,0.999911,...,1.000000,1.005618e+00,2.011236e+00,3.011236e+00,6.431818e+06,1150.387959,1150.182060,0.000179,0.004,0.004000
4742894,1,436,0,1378667.44,1,0.999853,2806215.27,NaN,NaN,0.999853,...,3.755102,6.296032e+12,3.755102e+00,-1.324058e+12,1.271066e+03,1.036057,1.037718,623.769312,0.001,0.001000
4742895,2,436,0,0.00,0,0.999373,4873746.26,NaN,NaN,0.999373,...,0.210526,6.836464e+12,2.105263e-01,-5.647514e+12,7.700395e+02,607.802775,2888.618567,3.746395,0.002,0.002000
4742896,3,436,0,7863029.87,1,0.999576,46879785.29,NaN,NaN,0.999576,...,0.063679,-2.031123e+12,6.367925e-02,NaN,2.280309e+03,4.975939,4.962933,381.581679,0.006,0.006000
4742897,4,436,0,4599490.24,-1,1.000425,15193377.29,NaN,NaN,0.999952,...,15.020833,6.257928e-01,6.964706e-01,8.854167e+00,6.414807e+03,2.303359,2.304545,1941.207108,0.004,0.004000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5237887,195,480,540,2440722.89,-1,1.000317,28280361.74,0.999734,0.999734,1.000317,...,9.636364,5.269212e+11,9.636364e+00,NaN,9.721828e+01,10.728671,12.183564,7.374204,0.004,0.004001
5237888,196,480,540,349510.47,-1,1.000643,9187699.11,1.000129,1.000386,1.000643,...,0.460227,NaN,4.602273e-01,7.926335e+11,8.040607e+01,61.205415,34.508349,1.292590,0.001,0.001001
5237889,197,480,540,0.00,0,0.995789,12725436.10,0.995789,0.995789,0.995789,...,10.750000,-2.822256e+11,1.075000e+01,NaN,7.684887e+01,756.887784,69.681820,9.722528,0.004,0.003983
5237890,198,480,540,1000898.84,1,0.999210,94773271.05,0.999210,0.999210,0.998970,...,5.315789,-9.251859e-13,-1.099231e-12,5.315789e+00,1.729011e+02,107.135719,283.295220,0.608175,0.006,0.005994


In [ ]:
import numpy as np
import lightgbm as lgb

lgb_params = {
    "objective": "mae",
    "n_estimators": 2200,
    "num_leaves": 256,
    "subsample": 0.6,
    "colsample_bytree": 0.8,
#         "learning_rate": 0.00871,
    "learning_rate": 0.01,
    'max_depth': 11,
    "n_jobs": 4,
    "device": "gpu",
    "verbosity": -1,
    "importance_type": "gain",
#         "reg_alpha": 0.1,
    "reg_alpha": 0.2,
    "reg_lambda": 3.25
}

feature_columns = list(df_train_feats.columns)
print(f"Features = {len(feature_columns)}")
#print(f"Feature length = {len(feature_columns)}")

num_folds = 5
fold_size = 480 // num_folds
gap = 5

models = []
models_cbt = []
scores = []

model_save_path = 'modelitos_para_despues' 
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

date_ids = df_train['date_id'].values

for i in range(num_folds):
    start = i * fold_size
    end = start + fold_size

    train_indices = (date_ids < start) | (date_ids > end)
    test_indices = (date_ids >= start) & (date_ids < end)

    
    gc.collect()
    
    df_fold_train = df_train_feats[train_indices]
    df_fold_train_target = df_train['target'][train_indices]
    df_fold_valid = df_train_feats[test_indices]
    df_fold_valid_target = df_train['target'][test_indices]

    print(f"Fold {i+1} Model Training")

    # Train a LightGBM model for the current fold
    lgb_model = lgb.LGBMRegressor(**lgb_params)
    lgb_model.fit(
        df_fold_train[feature_columns],
        df_fold_train_target,
        callbacks=[
            lgb.callback.log_evaluation(period=100),
        ],
    )
    
#         cbt_model = cbt.CatBoostRegressor(objective='MAE', iterations=5000,bagging_temperature=0.5,
#                                 colsample_bylevel = 0.7,learning_rate = 0.065,
#                                 od_wait = 25,max_depth = 7,l2_leaf_reg = 1.5,
#                                 min_data_in_leaf = 1000,random_strength=0.65,
#                                 verbose=0,use_best_model=True,task_type='CPU')
#         cbt_model.fit(
#             df_fold_train[feature_columns],
#             df_fold_train_target,
#             eval_set=[(df_fold_valid[feature_columns], df_fold_valid_target)]
#         )
    
#         models_cbt.append(cbt_model)

    models.append(lgb_model)
    # Save the model to a file
    model_filename = os.path.join(model_save_path, f'doblez_{i+1}.txt')
    lgb_model.booster_.save_model(model_filename)
    print(f"Model for fold {i+1} saved to {model_filename}")

    # Evaluate model performance on the validation set
    #------------LGB--------------#
    fold_predictions = lgb_model.predict(df_fold_valid[feature_columns])
    fold_score = mean_absolute_error(fold_predictions, df_fold_valid_target)
    scores.append(fold_score)
    print(f":LGB Fold {i+1} MAE: {fold_score}")
    #------------CBT--------------#
#         fold_predictions = cbt_model.predict(df_fold_valid[feature_columns])
#         fold_score_cbt = mean_absolute_error(fold_predictions, df_fold_valid_target)
#         scores.append(fold_score_cbt)
#         print(f"CBT Fold {i+1} MAE: {fold_score_cbt}")

    # Free up memory by deleting fold specific variables
    del df_fold_train, df_fold_train_target, df_fold_valid, df_fold_valid_target
    gc.collect()

# Calculate the average best iteration from all regular folds
average_best_iteration = int(np.mean([model.best_iteration_ for model in models]))

# Update the lgb_params with the average best iteration
final_model_params = lgb_params.copy()

# final_model_params['n_estimators'] = average_best_iteration
print(f"Training final model with average best iteration: {average_best_iteration}")

# Train the final model on the entire dataset
num_model = 1

for i in range(num_model):
    final_model = lgb.LGBMRegressor(**lgb_params)
    final_model.fit(
        df_train_feats[feature_columns],
        df_train['target'],
        callbacks=[
            lgb.callback.log_evaluation(period=100),
        ],
    )
    # Append the final model to the list of models
    models.append(final_model)
model_filename = os.path.join(model_save_path, f'doblez-conjunto.txt')
final_model.booster_.save_model(model_filename)

In [ ]:
import numpy as np
import lightgbm as lgb

lgb_params = {
    "objective": "mae",
    "n_estimators": 2200,
    "num_leaves": 256,
    "subsample": 0.6,
    "colsample_bytree": 0.8,
#         "learning_rate": 0.00871,
    "learning_rate": 0.01,
    'max_depth': 11,
    "n_jobs": 4,
    "device": "gpu",
    "verbosity": -1,
    "importance_type": "gain",
#         "reg_alpha": 0.1,
    "reg_alpha": 0.2,
    "reg_lambda": 3.25
}

feature_columns = list(df_train_feats.columns)
print(f"Features = {len(feature_columns)}")
#print(f"Feature length = {len(feature_columns)}")

num_folds = 5
fold_size = 480 // num_folds
gap = 5

models = []
models_cbt = []
scores = []

model_save_path = 'modelitos_para_despues' 
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

date_ids = df_train['date_id'].values
train_ind = len(df_train_feats[df_train_feats['date_id']<=450])

gc.collect()

df_fold_train = df_train_feats[:train_ind-gap]
df_fold_train_target = df_train['target'][:train_ind-gap]
df_fold_valid = df_train_feats[train_ind+1:]
df_fold_valid_target = df_train['target'][train_ind+1:]


# Train a LightGBM model for the current fold
lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(
    df_fold_train[feature_columns],
    df_fold_train_target,
    callbacks=[
        lgb.callback.log_evaluation(period=100),
    ],
)

# cbt_model = cbt.CatBoostRegressor(objective='MAE', iterations=5000,bagging_temperature=0.5,
#                         colsample_bylevel = 0.7,learning_rate = 0.065,
#                         od_wait = 25,max_depth = 7,l2_leaf_reg = 1.5,
#                         min_data_in_leaf = 1000,random_strength=0.65,
#                         verbose=0,use_best_model=True,task_type='CPU')
# cbt_model.fit(
#     df_fold_train[feature_columns],
#     df_fold_train_target,
#     eval_set=[(df_fold_valid[feature_columns], df_fold_valid_target)]
# )

# models_cbt.append(cbt_model)

models.append(lgb_model)
# Save the model to a file
model_filename = os.path.join(model_save_path, f'doblez_{i+1}.txt')
lgb_model.booster_.save_model(model_filename)
print(f"Model for fold {i+1} saved to {model_filename}")

# Evaluate model performance on the validation set
#------------LGB--------------#
fold_predictions = lgb_model.predict(df_fold_valid[feature_columns])
fold_score = mean_absolute_error(fold_predictions, df_fold_valid_target)
scores.append(fold_score)
print(f":LGB Fold MAE: {fold_score}")
#------------CBT--------------#
#         fold_predictions = cbt_model.predict(df_fold_valid[feature_columns])
#         fold_score_cbt = mean_absolute_error(fold_predictions, df_fold_valid_target)
#         scores.append(fold_score_cbt)
#         print(f"CBT Fold {i+1} MAE: {fold_score_cbt}")

# # Free up memory by deleting fold specific variables
# del df_fold_train, df_fold_train_target, df_fold_valid, df_fold_valid_target
# gc.collect()

# # Calculate the average best iteration from all regular folds
# average_best_iteration = int(np.mean([model.best_iteration_ for model in models]))

# # Update the lgb_params with the average best iteration
# final_model_params = lgb_params.copy()

# # final_model_params['n_estimators'] = average_best_iteration
# print(f"Training final model with average best iteration: {average_best_iteration}")

# # Train the final model on the entire dataset
# num_model = 1

# for i in range(num_model):
#     final_model = lgb.LGBMRegressor(**lgb_params)
#     final_model.fit(
#         df_train_feats[feature_columns],
#         df_train['target'],
#         callbacks=[
#             lgb.callback.log_evaluation(period=100),
#         ],
#     )
#     # Append the final model to the list of models
#     models.append(final_model)
# model_filename = os.path.join(model_save_path, f'doblez-conjunto.txt')
# final_model.booster_.save_model(model_filename)

Features = 182
Model for fold 182 saved to modelitos_para_despues\doblez_182.txt
:LGB Fold MAE: 6.1138607655080275


In [ ]:
import numpy as np
import lightgbm as lgb

lgb_params = {
    "objective": "mae",
    "n_estimators": 2200,
    "num_leaves": 256,
    "subsample": 0.6,
    "colsample_bytree": 0.8,
#         "learning_rate": 0.00871,
    "learning_rate": 0.01,
    'max_depth': 11,
    "n_jobs": 4,
    "device": "gpu",
    "verbosity": -1,
    "importance_type": "gain",
#         "reg_alpha": 0.1,
    "reg_alpha": 0.2,
    "reg_lambda": 3.25
}

feature_columns = list(df_train_feats.columns)
print(f"Features = {len(feature_columns)}")
#print(f"Feature length = {len(feature_columns)}")

num_folds = 5
fold_size = 480 // num_folds
gap = 5

models = []
models_cbt = []
scores = []

model_save_path = 'modelitos_para_despues' 
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

date_ids = df_train['date_id'].values
train_ind = len(df_train_feats[df_train_feats['seconds_in_bucket']<=450])

gc.collect()

df_fold_train = df_train_feats[:train_ind-gap]
df_fold_train_target = df_train['target'][:train_ind-gap]
df_fold_valid = df_train_feats[train_ind+1:]
df_fold_valid_target = df_train['target'][train_ind+1:]


cbt_model = cbt.CatBoostRegressor(objective='MAE', iterations=5000,bagging_temperature=0.5,
                        colsample_bylevel = 0.7,learning_rate = 0.065,
                        od_wait = 25,max_depth = 7,l2_leaf_reg = 1.5,
                        min_data_in_leaf = 1000,random_strength=0.65,
                        verbose=0,use_best_model=True,task_type='CPU')
cbt_model.fit(
    df_fold_train[feature_columns],
    df_fold_train_target,
    eval_set=[(df_fold_valid[feature_columns], df_fold_valid_target)]
)

models_cbt.append(cbt_model)


# Save the model to a file
model_filename = os.path.join(model_save_path, f'doblez_{i+1}.txt')
cbt_model.booster_.save_model(model_filename)
print(f"Model for fold saved to {model_filename}")

# Evaluate model performance on the validation set
#------------LGB--------------#
fold_predictions = cbt_model.predict(df_fold_valid[feature_columns])
fold_score = mean_absolute_error(fold_predictions, df_fold_valid_target)
scores.append(fold_score)
print(f":CBT Fold MAE: {fold_score}")

# # Free up memory by deleting fold specific variables
del df_fold_train, df_fold_train_target, df_fold_valid, df_fold_valid_target
gc.collect()

In [41]:
len(df_train_feats[df_train_feats['seconds_in_bucket']<=450])

3966779